In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- filter_concat_all ---
class _FilterDataset:
    def __init__(self, df):
        self._df = df
        self.x_columns = ["x1", "x2"]
        self.y_columns = ["y"]
        self.w_columns = ["w"]
        self.X = df[["x1", "x2"]]
        self.y = df[["y"]]
        self.w = df[["w"]]
    def to_pandas(self):
        return self._df.copy()
    def to_polars(self):
        return pl.from_pandas(self._df)
    def filter(self, mask):
        mask_list = mask.to_list() if hasattr(mask, "to_list") else list(mask)
        return pl.from_pandas(self._df.loc[mask_list].reset_index(drop=True))

_FILTER_DF = pd.DataFrame({"x1": [1., 2., 3., 4., 5.], "x2": [4., 5., 6., 7., 8.], "y": [0, 1, 0, 1, 0], "w": [1., 1., 1., 1., 1.]})
FIX_FILTER_CONCAT_ALL_DS = _FilterDataset(_FILTER_DF)
FIX_FILTER_CONCAT_ALL_FLGS = [pl.Series("m1", [True, False, True, True, False]), pl.Series("m2", [True, True, True, False, False])]
FIX_FILTER_CONCAT_ALL_FLGS_PD = [pd.Series([True, False, True, True, False]), pd.Series([True, True, True, False, False])]

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_filter_concat_all(ds, flgs):
    flg = pd.concat(flgs, axis=1).all(axis=1)
    df = ds.to_pandas().loc[flg]
    return df

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_filter_concat_all(ds, flgs):
    flg = pl.DataFrame(flgs).select(pl.all_horizontal()).to_series()
    df = ds.filter(flg)
    return df

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: filter_concat_all ===

try:
    _r = gen_filter_concat_all(FIX_FILTER_CONCAT_ALL_DS, FIX_FILTER_CONCAT_ALL_FLGS)
    print("✅ L1 smoke gen_filter_concat_all: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_filter_concat_all: {type(_e).__name__}: {_e}")

try:
    _rb = before_filter_concat_all(FIX_FILTER_CONCAT_ALL_DS, FIX_FILTER_CONCAT_ALL_FLGS_PD)
    print("✅ L1 smoke before_filter_concat_all: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_filter_concat_all: {type(_e).__name__}: {_e}")

try:
    _rb = before_filter_concat_all(FIX_FILTER_CONCAT_ALL_DS, FIX_FILTER_CONCAT_ALL_FLGS_PD)
    _rg = gen_filter_concat_all(FIX_FILTER_CONCAT_ALL_DS, FIX_FILTER_CONCAT_ALL_FLGS)
    compare(_rb, _rg, "filter_concat_all")
except Exception as _e:
    print(f"❌ L2 equivalence filter_concat_all: setup error — {type(_e).__name__}: {_e}")

try:
    _rb = before_filter_concat_all(FIX_FILTER_CONCAT_ALL_DS, [pd.Series([False] * len(_FILTER_DF)), pd.Series([False] * len(_FILTER_DF))])
    _rg = gen_filter_concat_all(FIX_FILTER_CONCAT_ALL_DS, [pl.Series("m1", [False] * len(_FILTER_DF)), pl.Series("m2", [False] * len(_FILTER_DF))])
    compare(_rb, _rg, "filter_concat_all no rows")
except Exception as _e:
    print(f"❌ L3 edge filter_concat_all no rows: {type(_e).__name__}: {_e}")
